# Session 6 — Your turn: From a linear model to a neural network

Two small jobs with numpy only: teach one sigmoid unit a function it can learn, then build by hand the network for the function it cannot.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np

X2 = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_and = np.array([0.0, 0.0, 0.0, 1.0])
y_xor = np.array([0.0, 1.0, 1.0, 0.0])
sigmoid = lambda z: 1 / (1 + np.exp(-z))
from aij_check import check, close, summary

## Task 1: One unit: AND yes, XOR no

Write `train_unit(X, y, steps=3000, lr=1.0)` that trains one sigmoid unit (`p = sigmoid(X @ w + b)`) by gradient descent on the cross-entropy loss, starting from `w = [0, 0]`, `b = 0`, and returns `(w, b, losses)`.
The gradient of the mean cross-entropy is `X.T @ (p - y) / len(y)` for `w` and `mean(p - y)` for `b`. Then train it on AND (`w_and, b_and, losses_and`) and on XOR (`w_xor, b_xor, losses_xor`).

In [ ]:
def train_unit(X, y, steps=3000, lr=1.0):
    w, b, losses = np.zeros(2), 0.0, []
    for _ in range(steps):
        p = ...                                   # predicted probabilities
        losses.append(...)                        # the mean cross-entropy loss
        ...                                       # one gradient step on w and b
    return w, b, losses


w_and, b_and, losses_and = train_unit(X2, y_and)
w_xor, b_xor, losses_xor = train_unit(X2, y_xor)

In [ ]:
check("the AND loss falls a lot", lambda: losses_and[-1] < 0.2 * losses_and[0])
check("the unit gets AND right", lambda: ((sigmoid(X2 @ w_and + b_and) > 0.5) == y_and.astype(bool)).all())
check("the unit cannot get XOR: it stays at probability 0.5", lambda: isinstance(losses_xor[-1], (float, np.floating)) and np.abs(sigmoid(X2 @ w_xor + b_xor) - 0.5).max() < 0.05, "No single straight line separates XOR.")
check("the XOR loss stays near ln 2 = 0.693", lambda: abs(losses_xor[-1] - np.log(2)) < 0.02)
summary()

## Task 2: Build the XOR network by hand

Choose the numbers of a 2-2-1 network with ReLU hidden units, `hidden = relu(X @ W1 + b1)` and `out = hidden @ W2 + b2`, so that `out` is **1 for the inputs (0,1) and (1,0) and 0 for (0,0) and (1,1)**.
(Hint: one hidden unit can ask 'is at least one input on?', the other 'are both on?', and the output subtracts them. With ReLU, `relu(x1 + x2)` is 0, 1, 1, 2 and `relu(x1 + x2 - 1)` is 0, 0, 0, 1.)

In [ ]:
relu = lambda z: np.maximum(0, z)
W1 = ...            # shape (2, 2)
b1 = ...            # shape (2,)
W2 = ...            # shape (2,)
b2 = ...            # a number
out = relu(X2 @ W1 + b1) @ W2 + b2
print(out)

In [ ]:
check("the output is 1 when exactly one input is on, 0 otherwise", lambda: close(out, y_xor, 1e-9), "The two hidden units should answer 'at least one on' and 'both on'; the output is their difference.")
check("the shapes are those of a 2-2-1 network", lambda: np.shape(W1) == (2, 2) and np.shape(b1) == (2,) and np.shape(W2) == (2,))
check("the network has 9 numbers", lambda: np.size(W1) + np.size(b1) + np.size(W2) + np.size(b2) == 9)
summary()